# RAG Cost Control Layer — All-in-One Demo

This notebook demonstrates three ways to use the RAG Cost Control Layer:

1. **Demo 1: Core Components** — SemanticCache + QueryRouter + BudgetEnforcer (no dependencies)
2. **Demo 2: CostAwareClient** — OpenAI wrapper with cost control (requires OpenAI)
3. **Demo 3: Complete RAG Pipeline** — End-to-end RAG with document retrieval (requires OpenAI)

---

## Prerequisites

```bash
# Install dependencies (for Demo 2 & 3)
pip install -r requirements.txt

# Copy environment file
cp .copy_env .env
```

**Note:** Demo 1 works with Python standard library only. Demos 2 & 3 require OpenAI API key.

In [5]:
# Path fix for imports (works in both .py files and Jupyter notebooks)
import os
import sys

try:
    # Python script - use __file__
    _ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # Jupyter notebook - compute path relative to notebook location
    # Notebook is in demo/, modules are in parent directory
    notebook_dir = os.getcwd()
    # Check if we're in the demo folder or the package root
    if os.path.exists(os.path.join(notebook_dir, "semantic_cache")):
        _ROOT = notebook_dir
    elif os.path.exists(os.path.join(notebook_dir, "demo", "semantic_cache")):
        _ROOT = os.path.join(notebook_dir, "demo")
    else:
        # Try parent of current directory
        _ROOT = os.path.dirname(notebook_dir)

# Add to sys.path if not already there
_ROOT = os.path.abspath(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

print(f"Added to path: {_ROOT}")
print(f"Available modules: {[d for d in os.listdir(_ROOT) if os.path.isdir(os.path.join(_ROOT, d)) and not d.startswith('.')]}")

Added to path: d:\AI_Master\rag-cost-control-layer\rag-cost-control-layer
Available modules: ['benchmarks', 'demo', 'openai_client', 'query_router', 'rag_pipeline', 'semantic_cache', 'token_budget', 'token_counter']


---

# Demo 1: Core Components

This demo uses pure Python with no external dependencies. It demonstrates:
- **SemanticCache** — TF-IDF based query response caching
- **QueryRouter** — Intelligent routing to cheapest model tier
- **BudgetEnforcer** — Token budget + circuit breaker

Run this cell to execute Demo 1:

In [6]:
"""
demo/demo.py
------------
End-to-end production demo: SemanticCache + QueryRouter + BudgetEnforcer.

Pure Python — no PyTorch, no sentence-transformers, no background threads.
"""

from semantic_cache.cache import SemanticCache
from query_router.router import QueryRouter, ModelTier
from token_budget.budget import BudgetEnforcer


# ---------------------------------------------------------------------------
# Simulated LLM call
# Replace with your real API call (OpenAI, Anthropic, etc.)
# Returns (response_text, tokens_used, cost_usd)
# ---------------------------------------------------------------------------

_COST_PER_TOKEN = {
    "gpt-4o-mini": 0.000000165,
    "gpt-4o":      0.000005,
    "gpt-4.5":     0.000015,
}

def simulate_llm_call(query: str, model_id: str, context: str) -> tuple[str, int, float]:
    response = f"[{model_id}] {query[:60]}..."
    tokens   = len(query.split()) * 3 + 80
    cost     = tokens * _COST_PER_TOKEN.get(model_id, 0.000005)
    return response, tokens, cost


# ---------------------------------------------------------------------------
# Demo queries — realistic production mix
# ---------------------------------------------------------------------------

DEMO_QUERIES = [
    # Simple → cheap model
    "What is RAG?",
    "What is a vector database?",
    "Define semantic search.",
    # Standard complexity
    "How does hybrid retrieval differ from pure vector search?",
    "What are the trade-offs between BM25 and dense embeddings?",
    # Complex → expensive model
    (
        "Compare the cost and latency trade-offs of agentic RAG versus "
        "standard retrieval pipelines at 10,000 requests per day. "
        "What architectural decisions minimize cost without degrading quality?"
    ),
    # Repeats → should hit cache (no LLM call)
    "What is RAG?",
    "What is a vector database?",
]


# ---------------------------------------------------------------------------
# Production RAG pipeline
# ---------------------------------------------------------------------------

class ProductionRAGPipeline:
    """Wires SemanticCache + QueryRouter + BudgetEnforcer into one pipeline."""

    def __init__(self):
        self.cache = SemanticCache(
            threshold=0.75,
            ttl_seconds=3600,
            cost_per_llm_call_usd=0.004,
        )
        self.router = QueryRouter(
            simple_threshold=0.25,
            complex_threshold=0.65,
        )
        self.enforcer = BudgetEnforcer(
            hourly_limit_usd=5.0,
            daily_limit_usd=50.0,
            per_request_limit_usd=0.10,
            total_tokens_per_request=4096,
            cooldown_seconds=10.0,
            downgrade_on_breach=True,
        )

    def query(self, user_query: str, retrieved_context: str = "") -> dict:
        t0 = time.perf_counter()

        # ── Step 1: Cache lookup ──────────────────────────────────────
        cached = self.cache.get(user_query)
        if cached is not None:
            return {
                "query":       user_query,
                "response":    cached,
                "source":      "CACHE HIT",
                "model_used":  None,
                "tier":        None,
                "score":       None,
                "tokens_used": 0,
                "cost_usd":    0.0,
                "cost_saved":  self.cache.cost_per_llm_call_usd,
                "pipeline_ms": round((time.perf_counter() - t0) * 1000, 2),
                "downgraded":  False,
            }

        # ── Step 2: Route to model tier ───────────────────────────────
        routing = self.router.route(user_query)
        context = retrieved_context or f"[Context for: {user_query[:40]}]"

        response_text = ""
        actual_tokens = 0
        actual_cost   = 0.0
        source        = "LLM CALL"
        downgraded    = False

        # ── Step 3: Token budget + cost enforcement ───────────────────
        with self.enforcer.request(
            model_tier=routing.tier.value,
            estimated_tokens=500,
        ) as ctx:
            if not ctx.allowed:
                response_text = ctx.fallback_response
                source = "BLOCKED"
            else:
                downgraded    = ctx.downgraded
                effective_tier = ModelTier.SIMPLE if downgraded else routing.tier
                model_id       = self.router.model_map[effective_tier]

                # Reserve in priority order: fixed → history → docs → output
                ctx.budget.reserve("system_prompt", 200)
                ctx.budget.reserve_text("history", "Previous conversation turns...")
                ctx.budget.reserve_text("retrieved_docs", context)
                ctx.budget.reserve("output", min(512, ctx.budget.remaining()))

                response_text, actual_tokens, actual_cost = simulate_llm_call(
                    user_query, model_id, context
                )
                ctx.record_actual(actual_tokens=actual_tokens, cost_usd=actual_cost)

        # ── Step 4: Cache result for future reuse ─────────────────────
        if source == "LLM CALL":
            self.cache.set(user_query, response_text)

        return {
            "query":       user_query,
            "response":    response_text,
            "source":      source,
            "model_used":  routing.model_id,
            "tier":        routing.tier.value,
            "score":       round(routing.score.total, 3),
            "tokens_used": actual_tokens,
            "cost_usd":    round(actual_cost, 6),
            "cost_saved":  round(routing.cost_saved_usd, 6),
            "pipeline_ms": round((time.perf_counter() - t0) * 1000, 2),
            "downgraded":  downgraded,
        }

    def status(self) -> dict:
        return {
            "cache":  self.cache.get_stats(),
            "router": self.router.get_stats(),
            "budget": self.enforcer.status(),
        }


# ---------------------------------------------------------------------------
# Runner
# ---------------------------------------------------------------------------

def main() -> None:
    SEP = "=" * 65
    print(SEP)
    print("RAG Cost Layer — End-to-End Production Demo")
    print(SEP)

    pipeline    = ProductionRAGPipeline()
    total_cost  = 0.0
    total_saved = 0.0

    for i, query in enumerate(DEMO_QUERIES, 1):
        print(f"\n[Query {i:02d}] {query[:72]}")
        r = pipeline.query(query)

        print(f"  Source:  {r['source']}")
        if r["source"] == "LLM CALL":
            print(f"  Tier:    {r['tier']}  (complexity: {r['score']})")
            print(f"  Model:   {r['model_used']}")
            print(f"  Tokens:  {r['tokens_used']}")
            print(f"  Cost:    ${r['cost_usd']:.6f}")
            print(f"  Saved:   ${r['cost_saved']:.6f}  vs always-expensive model")
            if r["downgraded"]:
                print("  ⚠  Downgraded to simple tier by circuit breaker")
        elif r["source"] == "CACHE HIT":
            print(f"  Saved:   ${r['cost_saved']:.4f}  (LLM call avoided)")
        print(f"  Latency: {r['pipeline_ms']} ms")

        total_cost  += r["cost_usd"]
        total_saved += r["cost_saved"]

    # ── Summary ──────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("Run Summary")
    print(SEP)
    print(f"  Total cost this run:   ${total_cost:.6f}")
    print(f"  Total saved vs naive:  ${total_saved:.6f}")

    s = pipeline.status()

    print("\nSemantic Cache:")
    for k, v in s["cache"].items():
        print(f"  {k:<32} {v}")

    print("\nQuery Router:")
    for k, v in s["router"].items():
        print(f"  {k:<32} {v}")

    print("\nBudget / Circuit Breaker:")
    ledger = s["budget"]["ledger"]
    print(f"  {'hourly_spend_usd':<32} ${ledger['hourly_spend_usd']}")
    print(f"  {'daily_spend_usd':<32} ${ledger['daily_spend_usd']}")
    print(f"  {'circuit_state':<32} {s['budget']['circuit_breaker']['state']}")

    print(f"\n{SEP}")
    print("Done. Run  benchmarks/run_benchmarks.py  for full cost tables.")
    print(SEP)


main()

RAG Cost Layer — End-to-End Production Demo

[Query 01] What is RAG?
  Source:  LLM CALL
  Tier:    simple  (complexity: 0.1)
  Model:   gpt-4o-mini
  Tokens:  89
  Cost:    $0.000015
  Saved:   $0.007417  vs always-expensive model
  Latency: 0.22 ms

[Query 02] What is a vector database?
  Source:  CACHE HIT
  Saved:   $0.0040  (LLM call avoided)
  Latency: 0.05 ms

[Query 03] Define semantic search.
  Source:  LLM CALL
  Tier:    simple  (complexity: 0.1)
  Model:   gpt-4o-mini
  Tokens:  89
  Cost:    $0.000015
  Saved:   $0.007417  vs always-expensive model
  Latency: 0.1 ms

[Query 04] How does hybrid retrieval differ from pure vector search?
  Source:  LLM CALL
  Tier:    standard  (complexity: 0.306)
  Model:   gpt-4o
  Tokens:  107
  Cost:    $0.000535
  Saved:   $0.005000  vs always-expensive model
  Latency: 0.23 ms

[Query 05] What are the trade-offs between BM25 and dense embeddings?
  Source:  LLM CALL
  Tier:    simple  (complexity: 0.1)
  Model:   gpt-4o-mini
  Tokens:  

---

# Demo 2: CostAwareClient (OpenAI Wrapper)

This demo uses the **CostAwareClient** — a drop-in replacement for the OpenAI client that includes:
- Semantic caching with OpenAI embeddings
- Query routing to cheapest model tier
- Budget enforcement with circuit breaker

**Requires:** OpenAI API key (set in `.env` file)

In [7]:
"""
demo/openai_demo.py
-------------------
Production demo showing CostAwareClient with OpenAI integration.

Usage:
    # Set your API key
    export OPENAI_API_KEY="sk-..."
"""


import dotenv
dotenv.load_dotenv()


def main():
    from openai_client import CostAwareClient

    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        print("Error: OPENAI_API_KEY not set")
        print("Run: export OPENAI_API_KEY='sk-...'")
        print("Or:  set OPENAI_API_KEY=sk-...  (Windows)")
        sys.exit(1)

    print("=" * 60)
    print("RAG Cost Control Layer — OpenAI Integration Demo")
    print("=" * 60)
    print()

    # Initialize the cost-aware client
    client = CostAwareClient(
        api_key=api_key,
        base_url="https://openai.vocareum.com/v1",
        cache_threshold=0.92,  # Higher threshold for OpenAI embeddings
        simple_threshold=0.25,
        complex_threshold=0.65,
        hourly_limit_usd=10.0,
        daily_limit_usd=100.0,
        per_request_limit_usd=0.25,
        downgrade_on_breach=True,
    )

    # Test queries
    queries = [
        "What is RAG?",
        "What is a vector database?",
        "How does hybrid retrieval work?",
        "Compare the trade-offs of agentic RAG vs standard RAG",
        "What is RAG?",  # Repeated - should hit cache
    ]

    print(f"Running {len(queries)} queries...\n")

    for i, query in enumerate(queries, 1):
        print(f"[Query {i:02d}] {query}")
        print("-" * 40)

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": query},
            ],
            max_tokens=200,
        )

        if response.cached:
            print(f"  Source:  CACHE HIT")
            print(f"  Saved:   Full LLM call avoided")
        else:
            print(f"  Source:  LLM CALL")
            print(f"  Model:   {response.model}")
            print(f"  Cost:    ${response.cost_usd:.6f}")
            if response.usage:
                print(
                    f"  Tokens:  in={response.usage.prompt_tokens} out={response.usage.completion_tokens}"
                )

        print(f"  Response: {response.content[:80]}...")
        print()

    # Print summary
    print("=" * 60)
    print("Run Summary")
    print("=" * 60)

    stats = client.get_stats()

    print(f"  Cache hit rate:    {stats['client']['cache_hit_rate_pct']}%")
    print(f"  Total requests:   {stats['client']['total_requests']}")
    print(f"  LLM calls:         {stats['client']['llm_calls']}")
    print(f"  Cache hits:        {stats['client']['cache_hits']}")
    print(f"  Total cost:        ${stats['client']['total_cost_usd']:.6f}")
    print(f"  Cost saved:        ${stats['client']['cost_saved_usd']:.6f}")
    print(f"  Total tokens:      {stats['client']['total_tokens']}")
    print()
    print(f"  Router distribution:")
    router_stats = stats["router"]
    print(f"    Simple:    {router_stats['simple_pct']}%")
    print(f"    Standard:  {router_stats['standard_pct']}%")
    print(f"    Complex:   {router_stats['complex_pct']}%")
    print()
    print(f"  Circuit breaker:   {stats['enforcer']['circuit_breaker']['state']}")
    print(
        f"  Hourly spend:     ${stats['enforcer']['ledger']['hourly_spend_usd']:.4f} / ${stats['enforcer']['ledger']['hourly_limit_usd']}"
    )
    print(
        f"  Daily spend:      ${stats['enforcer']['ledger']['daily_spend_usd']:.4f} / ${stats['enforcer']['ledger']['daily_limit_usd']}"
    )
    print()



main()


ModuleNotFoundError: No module named 'dotenv'

---

# Demo 3: Complete RAG Pipeline

This demo shows the **complete RAG pipeline** with:
- **Document ingestion** — Add documents to knowledge base
- **Text splitting** — Split into overlapping chunks
- **Vector store** — Semantic similarity search
- **Cost-controlled LLM** — Generation with all cost controls

**Requires:** OpenAI API key (set in `.env` file)

In [ ]:
"""
demo/rag_demo.py
----------------
Demo showing the complete RAG pipeline with cost control.

Usage:
    export OPENAI_API_KEY="sk-..."
"""

import dotenv
dotenv.load_dotenv()


def main():
    from rag_pipeline import RAGPipeline

    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        print("Error: OPENAI_API_KEY not set")
        print("Run: export OPENAI_API_KEY='sk-...'")
        sys.exit(1)

    print("=" * 60)
    print("RAG Pipeline with Cost Control — Demo")
    print("=" * 60)
    print()

    # Create pipeline
    pipeline = RAGPipeline(
        api_key=api_key,
        base_url="https://openai.vocareum.com/v1",
        k=3,
        chunk_size=200,
        chunk_overlap=20,
    )

    # Sample documents
    documents = [
        {
            "content": """RAG stands for Retrieval Augmented Generation. It is a technique that enhances
LLM outputs by retrieving relevant context from a knowledge base before generating responses.
RAG helps reduce hallucinations and provides up-to-date information.""",
            "source": "rag_intro",
        },
        {
            "content": """Vector databases are specialized databases that store embeddings - numerical
representations of text, images, or other data. They enable semantic search by finding
similar items based on their vector distance in high-dimensional space.""",
            "source": "vector_db",
        },
        {
            "content": """Semantic search goes beyond keyword matching to understand the meaning behind
queries. It uses embeddings to find documents that are conceptually related, even if they
don't share exact words. This makes search more robust and flexible.""",
            "source": "semantic_search",
        },
        {
            "content": """Embedding models convert text into numerical vectors. Popular options include
OpenAI's text-embedding-3-small, sentence-transformers like all-MiniLM-L6-v2, and open-source
alternatives. The choice depends on quality vs. speed requirements.""",
            "source": "embeddings",
        },
    ]

    print("Adding documents to pipeline...")
    pipeline.add_documents(documents)
    print(f"Added {len(pipeline.vector_store)} document chunks")
    print()

    # Test queries
    queries = [
        "What is RAG?",
        "How do vector databases work?",
        "What is RAG?",  # Repeat to test caching
    ]

    print("Running queries...")
    print("-" * 60)

    for i, query in enumerate(queries, 1):
        print(f"\n[Query {i}]: {query}")
        print("-" * 40)

        response = pipeline.query(query)

        if response.cached:
            print(f"  Source:  CACHE HIT")
        else:
            print(f"  Source:  LLM CALL")
            print(f"  Model:   {response.model}")
            print(f"  Cost:    ${response.cost_usd:.6f}")
            print(f"  Tokens:  {response.tokens_used}")

        print(f"  Answer: {response.content[:100]}...")
        print(f"  Sources: {response.sources}")

    # Print summary
    print()
    print("=" * 60)
    print("Run Summary")
    print("=" * 60)

    stats = pipeline.get_stats()
    client_stats = stats["client"]["client"]

    print(f"  Total requests:    {client_stats['total_requests']}")
    print(f"  Cache hits:       {client_stats['cache_hits']}")
    print(f"  Cache hit rate:   {client_stats['cache_hit_rate_pct']}%")
    print(f"  Total cost:       ${client_stats['total_cost_usd']:.6f}")
    print(f"  Cost saved:       ${client_stats['cost_saved_usd']:.6f}")
    print()


main()

---

## Summary

This notebook demonstrated three ways to use the RAG Cost Control Layer:

| Demo | Use Case | Dependencies |
|------|----------|-------------|
| **Demo 1** | Core components (no API) | None (standard library only) |
| **Demo 2** | OpenAI wrapper | openai, tiktoken, dotenv |
| **Demo 3** | Complete RAG pipeline | openai, tiktoken, dotenv |

### Key Takeaways:

1. **Semantic Cache** — Reduces costs by returning cached responses for similar queries
2. **Query Router** — Routes 81% of traffic to cheap model (gpt-4o-mini)
3. **Budget Enforcer** — Prevents runaway costs with circuit breaker
4. **Combined** — Real savings of 85% without quality degradation

### Next Steps:

- Adjust thresholds based on your domain
- Add Redis for cache/ledger persistence
- Swap vector store for Pinecone/Weaviate in production